# 03 — Feature Engineering
## Context-Aware Neural Recommendation Engine
### H&M Personalized Fashion Recommendations

---

**Notebook Purpose:** Experiment with feature engineering, understand recommendation signals, verify outputs visually, and test logic before promoting it to production scripts in `src/feature_engineering/`.

**Workflow Position:**
```
01_data_understanding.ipynb
02_preprocessing.ipynb
→ 03_feature_engineering.ipynb   ← YOU ARE HERE
04_model_training.ipynb (coming next)
```

---

## Section 1: Introduction — What Is Feature Engineering?

### Why Do Recommendation Models Need Features?

Raw data tells us facts: *"Customer A bought Article B on Date C."*  
Features tell us **signals**: *"Customer A is a frequent buyer who last shopped 3 days ago and prefers blue dresses."*

Machine learning models don't understand raw data — they understand numbers. Feature engineering is the process of transforming raw data into meaningful numerical representations that models can learn from.

### Three Types of Features in a Recommendation System

| Feature Type | Question It Answers | Example Features |
|---|---|---|
| **User Features** | Who is this customer? | purchase_count, recency_score, active_status |
| **Item Features** | What is this product? | popularity_score, colour_group, garment_type |
| **Interaction Features** | Who bought what? | (customer_id, article_id, label, recency_weight) |

### How This Connects to Two-Tower Models (TensorFlow Recommenders)

The **Two-Tower architecture** has two neural networks:
- **User Tower**: Takes user features → outputs a user embedding (dense vector)
- **Item Tower**: Takes item features → outputs an item embedding (dense vector)

The model learns by maximizing the **dot product** between user and item embeddings for items the user actually purchased.

```
User Features ──→ [User Tower] ──→ User Embedding ─┐
                                                    ├─→ dot_product → score
Item Features ──→ [Item Tower] ──→ Item Embedding ─┘
```

**The richer our features, the more the towers have to learn from.**

---
## Section 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# Display settings — show more columns and rows in output
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("Libraries imported successfully.")
print(f"pandas version : {pd.__version__}")
print(f"numpy  version : {np.__version__}")

---
## Section 3: Load Cleaned Datasets

We load the three cleaned datasets generated by our preprocessing pipeline (`02_preprocessing.ipynb`).

These files live in `../data/processed/` and have already been:
- De-duplicated
- Missing values handled
- Data types standardized

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

CUSTOMERS_PATH    = os.path.join(PROCESSED_DIR, 'customers_cleaned.csv')
ARTICLES_PATH     = os.path.join(PROCESSED_DIR, 'articles_cleaned.csv')
TRANSACTIONS_PATH = os.path.join(PROCESSED_DIR, 'transactions_cleaned.csv')

# ── Load data ─────────────────────────────────────────────────────────────────
# parse_dates=['t_dat'] tells pandas to read the date column as datetime
# instead of a string — required for recency calculations
customers    = pd.read_csv(CUSTOMERS_PATH)
articles     = pd.read_csv(ARTICLES_PATH)
transactions = pd.read_csv(TRANSACTIONS_PATH, parse_dates=['t_dat'])

print("Datasets loaded successfully!")
print(f"  customers    : {customers.shape[0]:,} rows × {customers.shape[1]} columns")
print(f"  articles     : {articles.shape[0]:,} rows × {articles.shape[1]} columns")
print(f"  transactions : {transactions.shape[0]:,} rows × {transactions.shape[1]} columns")

In [ ]:
# Quick sanity check — preview each dataset
print("=" * 50)
print("CUSTOMERS preview:")
display(customers.head(3))

print("\nARTICLES preview:")
display(articles.head(3))

print("\nTRANSACTIONS preview:")
display(transactions.head(3))

---
## Section 4: User Feature Engineering

### What Are User Features?

User features answer: **"What kind of shopper is this person?"**

We combine:
1. **Demographic info** from `customers_cleaned.csv` (age, club membership, etc.)
2. **Behavioral signals** derived from `transactions_cleaned.csv` (how often, how recently, how much they buy)

Behavioral signals are usually **more powerful** than demographics for recommendations because they reflect actual revealed preferences.

### 4.1 — Purchase Count

**What it is:** Total number of purchases a customer has made.  
**Why it matters:** High-count customers are the most engaged. They're likely to respond well to personalized recommendations. Low-count customers (or new customers with 0 purchases) are "cold start" users who need popularity-based recommendations instead.

In [ ]:
# Count total purchases per customer
# .groupby('customer_id') groups all rows for the same customer together
# .size() counts how many rows are in each group
purchase_count = (
    transactions
    .groupby('customer_id')
    .size()
    .reset_index(name='purchase_count')
)

print(f"Purchase count computed for {len(purchase_count):,} customers.")
print("\nDistribution:")
print(purchase_count['purchase_count'].describe())

print("\nSample:")
display(purchase_count.head(5))

### 4.2 — Recency Features

**What it is:** How recently a customer made their last purchase.  
**Why it matters:** In fashion, preferences change quickly. A customer who bought last week has active, current preferences. Someone who last bought 2 years ago may have very different needs today — or may have churned entirely.

**RFM Framework:** Recency is the **R** in the classic marketing framework:  
- **R**ecency — How recently did they buy?  
- **F**requency — How often do they buy?  
- **M**onetary — How much do they spend?

In [ ]:
# Use the last date in the dataset as our "today" reference
# In production, this would be datetime.today()
REFERENCE_DATE = pd.Timestamp('2020-09-23')
print(f"Reference date (simulated 'today'): {REFERENCE_DATE.date()}")

# Find each customer's most recent purchase date
recency = (
    transactions
    .groupby('customer_id')['t_dat']
    .max()                           # Most recent purchase
    .reset_index()
    .rename(columns={'t_dat': 'last_purchase_date'})
)

# Calculate days since that purchase
recency['days_since_purchase'] = (
    REFERENCE_DATE - recency['last_purchase_date']
).dt.days

# Create a normalized recency score: 1.0 = bought today, 0.0 = never/very long ago
max_days = recency['days_since_purchase'].max()
recency['recency_score'] = 1 - (recency['days_since_purchase'] / max_days)

print("\nRecency features:")
display(recency.head(5))

print("\nDays since purchase distribution:")
print(recency['days_since_purchase'].describe())

### 4.3 — Purchase Frequency

**What it is:** Average number of purchases per week across the customer's history.  
**Why it matters:** Frequency separates casual browsers from habitual buyers. A weekly shopper is a very different recommendation target than someone who buys once a year during a sale.

In [ ]:
# Calculate the span between first and last purchase (in weeks)
span = transactions.groupby('customer_id')['t_dat'].agg(
    first_purchase='min',
    last_purchase='max'
)

# Convert to weeks — clip(lower=1) prevents division by zero for single-purchase customers
span['span_weeks'] = (
    (span['last_purchase'] - span['first_purchase']).dt.days / 7
).clip(lower=1)

# Merge with purchase counts
frequency = span.reset_index().merge(purchase_count, on='customer_id')
frequency['purchase_frequency'] = frequency['purchase_count'] / frequency['span_weeks']

frequency = frequency[['customer_id', 'purchase_frequency']]

print("Purchase frequency stats (purchases per week):")
print(frequency['purchase_frequency'].describe())

print("\nSample:")
display(frequency.head(5))

### 4.4 — Active Status

**What it is:** Binary flag (1 = active, 0 = inactive) based on recency threshold.  
**Why it matters:** Active vs. inactive is a clean, interpretable feature that can help the model quickly segment recommendation strategy. Some models even train separate systems for active and inactive users.

In [ ]:
# A customer is considered 'active' if they purchased within the last 90 days
ACTIVE_THRESHOLD_DAYS = 90

recency['active_status'] = (recency['days_since_purchase'] <= ACTIVE_THRESHOLD_DAYS).astype(int)

active_count   = recency['active_status'].sum()
inactive_count = len(recency) - active_count

print(f"Active customers   (bought within {ACTIVE_THRESHOLD_DAYS} days): {active_count:,} ({100*active_count/len(recency):.1f}%)")
print(f"Inactive customers                                    : {inactive_count:,} ({100*inactive_count/len(recency):.1f}%)")

### 4.5 — Spend Features

**What it is:** Average and total spend per customer.  
**Why it matters:** Price sensitivity varies widely. The model can use spend features to recommend products in the right price range — avoiding recommending premium items to budget shoppers, and basics to luxury buyers.

In [ ]:
if 'price' in transactions.columns:
    spend = transactions.groupby('customer_id')['price'].agg(
        total_spend='sum',
        avg_spend_per_transaction='mean'
    ).reset_index()
    
    print("Spend features:")
    print(spend[['total_spend', 'avg_spend_per_transaction']].describe())
    display(spend.head(5))
else:
    print("'price' column not found in transactions — spend features skipped.")
    spend = transactions[['customer_id']].drop_duplicates()

### 4.6 — Merge All User Features

In [ ]:
# Start with the base customer table (demographics)
user_features = customers.copy()

# Merge behavioral features one by one
user_features = user_features.merge(purchase_count, on='customer_id', how='left')
user_features = user_features.merge(recency,        on='customer_id', how='left')
user_features = user_features.merge(frequency,      on='customer_id', how='left')
user_features = user_features.merge(spend,          on='customer_id', how='left')

# Fill NaN for customers with no transaction history
# (cold-start users — they exist in the customer table but never transacted)
user_features['purchase_count']           = user_features['purchase_count'].fillna(0).astype(int)
user_features['days_since_purchase']      = user_features['days_since_purchase'].fillna(9999)
user_features['recency_score']            = user_features['recency_score'].fillna(0)
user_features['purchase_frequency']       = user_features['purchase_frequency'].fillna(0)
user_features['active_status']            = user_features['active_status'].fillna(0).astype(int)

print(f"User features shape: {user_features.shape}")
print(f"\nColumns: {list(user_features.columns)}")
print("\nSample:")
display(user_features.head(5))

---
## Section 5: Item Feature Engineering

### What Are Item Features?

Item features answer: **"What is this product, and how has it performed?"**

We combine:
1. **Catalog attributes** from `articles_cleaned.csv` (color, type, category)
2. **Popularity signals** from `transactions_cleaned.csv` (how many people bought it)

Catalog attributes enable **content-based filtering** — recommending similar products.  
Popularity signals enable **collaborative filtering** — recommending what many people like.

### 5.1 — Product Popularity

In [ ]:
# Total purchases and unique buyers per article
popularity = transactions.groupby('article_id').agg(
    total_purchases=('customer_id', 'count'),
    unique_buyers=('customer_id', 'nunique')
).reset_index()

# Normalize popularity score to [0, 1]
max_purchases = popularity['total_purchases'].max()
popularity['popularity_score'] = popularity['total_purchases'] / max_purchases

print(f"Popularity computed for {len(popularity):,} articles.")
print("\nTop 5 most popular articles:")
display(popularity.sort_values('total_purchases', ascending=False).head(5))

### 5.2 — Recent Popularity

**Why it matters:** Fashion trends change fast. An item that sold well last winter may be completely irrelevant this summer. Recent popularity (last 30 days) is often a stronger signal than all-time popularity.

In [ ]:
RECENT_WINDOW_DAYS = 30
cutoff_date = REFERENCE_DATE - pd.Timedelta(days=RECENT_WINDOW_DAYS)

# Filter transactions to recent window only
recent_transactions = transactions[transactions['t_dat'] >= cutoff_date]

recent_pop = recent_transactions.groupby('article_id').agg(
    recent_purchases=('customer_id', 'count')
).reset_index()

max_recent = recent_pop['recent_purchases'].max()
recent_pop['recent_popularity_score'] = recent_pop['recent_purchases'] / max_recent

print(f"Recent popularity computed (last {RECENT_WINDOW_DAYS} days) for {len(recent_pop):,} articles.")
print("\nTop 5 trending articles (recent 30 days):")
display(recent_pop.sort_values('recent_purchases', ascending=False).head(5))

### 5.3 — Catalog Attribute Features

**Why it matters:** These features allow the model to find product similarity purely from catalog data — useful for cold-start items (newly listed products with no purchase history yet).

In [ ]:
# Select key categorical attributes from the article catalog
catalog_cols = ['article_id']
optional_cols = [
    'product_type_name',
    'product_group_name',
    'colour_group_name',
    'section_name',
    'garment_group_name',
    'index_name',
    'index_group_name',
]

# Only keep columns that exist in our dataset
available_cols = [c for c in optional_cols if c in articles.columns]
catalog = articles[catalog_cols + available_cols].copy()

print(f"Catalog attributes selected: {available_cols}")
print(f"\nCatalog shape: {catalog.shape}")
display(catalog.head(5))

In [ ]:
# Look at value distributions for key categorical features
if 'product_group_name' in catalog.columns:
    print("Product Group distribution (top 10):")
    print(catalog['product_group_name'].value_counts().head(10))

if 'colour_group_name' in catalog.columns:
    print("\nColour Group distribution (top 10):")
    print(catalog['colour_group_name'].value_counts().head(10))

### 5.4 — Article Age Feature

In [ ]:
# When was each article first sold? (proxy for 'age' in catalog)
first_sold = (
    transactions
    .groupby('article_id')['t_dat']
    .min()
    .reset_index()
    .rename(columns={'t_dat': 'first_sold_date'})
)

first_sold['article_age_days'] = (REFERENCE_DATE - first_sold['first_sold_date']).dt.days

print("Article age stats (days since first sold):")
print(first_sold['article_age_days'].describe())
display(first_sold.head(5))

### 5.5 — Merge All Item Features

In [ ]:
# Merge catalog + all behavioral/popularity features
item_features = catalog.copy()
item_features = item_features.merge(popularity,   on='article_id', how='left')
item_features = item_features.merge(recent_pop,   on='article_id', how='left')
item_features = item_features.merge(first_sold,   on='article_id', how='left')

# Fill 0 for items never purchased (catalog-only items)
for col in ['total_purchases', 'unique_buyers', 'popularity_score', 'recent_purchases', 'recent_popularity_score']:
    if col in item_features.columns:
        item_features[col] = item_features[col].fillna(0)

item_features['article_age_days'] = item_features['article_age_days'].fillna(
    item_features['article_age_days'].max()
)

print(f"Item features shape: {item_features.shape}")
print(f"\nColumns: {list(item_features.columns)}")
display(item_features.head(5))

---
## Section 6: Interaction Feature Engineering

### Why Interaction Data Is the Heart of Recommendation Systems

All recommendation learning ultimately comes down to one question:

> *"Given user U, which items should I recommend?"*

The model answers this by learning from **historical interactions** — the (user, item) pairs where a purchase actually happened. This is called **implicit feedback** because the user didn't explicitly rate anything; they just bought it.

The interaction table has three key components:

| Component | Description |
|---|---|
| `customer_id` | Who made the interaction |
| `article_id` | What item was interacted with |
| `label` | 1 = purchased (positive), 0 = not purchased (negative) |

**Why do we need negatives?**  
Without negatives, the model has no contrast — it can't learn what users *don't* like. We randomly sample items the user never bought as "implicit negatives".

### 6.1 — Build Positive Interactions

In [ ]:
# Aggregate: for each (customer, article) pair, keep most recent date and count purchases
pair_counts = (
    transactions
    .groupby(['customer_id', 'article_id'])
    .size()
    .reset_index(name='pair_purchase_count')
)

most_recent = (
    transactions
    .groupby(['customer_id', 'article_id'])['t_dat']
    .max()
    .reset_index()
)

positives = most_recent.merge(pair_counts, on=['customer_id', 'article_id'])
positives['label'] = 1

print(f"Positive (customer, article) pairs: {len(positives):,}")
print(f"Unique customers in interactions : {positives['customer_id'].nunique():,}")
print(f"Unique articles  in interactions : {positives['article_id'].nunique():,}")
display(positives.head(5))

---
## Section 7: Recency Features for Interactions

### Why Recent Behavior Matters More

Think about your own shopping habits. What you bought 3 years ago doesn't reflect what you want today nearly as well as what you bought last month.

**Exponential decay** is the standard way to weight recency:
```
weight = exp(−days_since_purchase / decay_scale)
```
- Purchase today  → weight ≈ 1.0  
- Purchase 90 days ago → weight ≈ 0.37  
- Purchase 365 days ago → weight ≈ 0.02  

These weights will be used during model training to put more emphasis on recent interactions in the loss function.

In [ ]:
# Calculate days since each purchase
positives['days_since_purchase'] = (REFERENCE_DATE - positives['t_dat']).dt.days

# Apply exponential decay weighting
# decay_scale = 90 means purchases 90 days ago are weighted at ~37% of today's weight
DECAY_SCALE = 90
positives['recency_weight'] = np.exp(-positives['days_since_purchase'] / DECAY_SCALE)

print("Recency weight distribution:")
print(positives['recency_weight'].describe())

# Verify the math
print("\nManual verification:")
print(f"  Weight for purchase 0 days ago   : {np.exp(-0/DECAY_SCALE):.4f}")
print(f"  Weight for purchase 90 days ago  : {np.exp(-90/DECAY_SCALE):.4f}")
print(f"  Weight for purchase 365 days ago : {np.exp(-365/DECAY_SCALE):.4f}")

### 7.1 — Negative Interaction Sampling

**What are negative samples?**  
Items the user never bought are treated as implicit negatives (label = 0).

**Why do we sample instead of using all negatives?**  
Each user has ~100,000 potential items they didn't buy. Using all would make the dataset enormous and heavily imbalanced. We sample 1 negative per positive (1:1 ratio) as a starting point.

> **Note:** For the full dataset this can be slow. In production, this step runs in `build_interactions.py`. Here we demonstrate the logic on a sample.

In [ ]:
# Work on a small sample to keep the notebook fast
SAMPLE_CUSTOMERS = 1000
NEGATIVE_RATIO   = 1      # 1 negative per positive
RANDOM_SEED      = 42

rng = np.random.default_rng(RANDOM_SEED)

# Sample customers and get their positives
sampled_customers = positives['customer_id'].drop_duplicates().sample(
    n=min(SAMPLE_CUSTOMERS, positives['customer_id'].nunique()),
    random_state=RANDOM_SEED
)
positives_sample = positives[positives['customer_id'].isin(sampled_customers)]

# All article IDs available for sampling
all_article_ids = transactions['article_id'].unique()

# Build a set of known positives for fast lookup
positive_set = set(zip(positives_sample['customer_id'], positives_sample['article_id']))

negative_rows = []
for customer_id in sampled_customers:
    n_pos = positives_sample[positives_sample['customer_id'] == customer_id].shape[0]
    n_neg = n_pos * NEGATIVE_RATIO
    
    # Sample more candidates than needed to filter out accidental positives
    candidates = rng.choice(all_article_ids, size=min(n_neg * 3, len(all_article_ids)), replace=False)
    
    sampled = 0
    for article_id in candidates:
        if sampled >= n_neg:
            break
        if (customer_id, article_id) not in positive_set:
            negative_rows.append({'customer_id': customer_id, 'article_id': article_id, 'label': 0})
            sampled += 1

negatives_sample = pd.DataFrame(negative_rows)
negatives_sample['recency_weight'] = 0.0
negatives_sample['days_since_purchase'] = np.nan

print(f"Positive interactions in sample : {len(positives_sample):,}")
print(f"Negative interactions in sample : {len(negatives_sample):,}")
print(f"Ratio  (pos:neg)                : 1:{NEGATIVE_RATIO}")

---
## Section 8: Merge and Prepare Final Feature Tables

Now we bring everything together into three clean, model-ready datasets:
1. `user_features` — one row per customer
2. `item_features` — one row per article
3. `interaction_features` — one row per (customer, article) pair with label

In [ ]:
# ── user_features: Already assembled in Section 4 ─────────────────────────────
print("USER FEATURES")
print(f"  Shape    : {user_features.shape}")
print(f"  Columns  : {list(user_features.columns)}")
print(f"  Nulls    : {user_features.isnull().sum().sum()}")
display(user_features.head(3))

In [ ]:
# ── item_features: Already assembled in Section 5 ─────────────────────────────
print("ITEM FEATURES")
print(f"  Shape    : {item_features.shape}")
print(f"  Columns  : {list(item_features.columns)}")
print(f"  Nulls    : {item_features.isnull().sum().sum()}")
display(item_features.head(3))

In [ ]:
# ── interaction_features: Combine positives + negatives ───────────────────────
# For the full dataset, use the production script: build_interactions.py
# Here we use the full positives + sample negatives for demonstration

interaction_features = pd.concat(
    [positives, negatives_sample],
    ignore_index=True
).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("INTERACTION FEATURES")
print(f"  Shape           : {interaction_features.shape}")
print(f"  Columns         : {list(interaction_features.columns)}")
print(f"  Label 1 (pos)   : {(interaction_features.label == 1).sum():,}")
print(f"  Label 0 (neg)   : {(interaction_features.label == 0).sum():,}")
display(interaction_features.head(5))

---
## Section 9: Save Final Feature Datasets

We save the three feature tables to `../data/processed/` so that:
1. Other notebooks can load them without re-running this pipeline
2. The model training script can directly consume them
3. The production pipeline (`run_feature_pipeline.py`) matches this exact format

In [ ]:
# Output paths
USER_FEATURES_OUT    = os.path.join(PROCESSED_DIR, 'user_features.csv')
ITEM_FEATURES_OUT    = os.path.join(PROCESSED_DIR, 'item_features.csv')
INTERACTION_OUT      = os.path.join(PROCESSED_DIR, 'interaction_features.csv')

# Save each dataset
user_features.to_csv(USER_FEATURES_OUT, index=False)
print(f"✅ user_features saved → {USER_FEATURES_OUT}")
print(f"   Shape: {user_features.shape}")

item_features.to_csv(ITEM_FEATURES_OUT, index=False)
print(f"\n✅ item_features saved → {ITEM_FEATURES_OUT}")
print(f"   Shape: {item_features.shape}")

interaction_features.to_csv(INTERACTION_OUT, index=False)
print(f"\n✅ interaction_features saved → {INTERACTION_OUT}")
print(f"   Shape: {interaction_features.shape}")

print("\nAll feature datasets saved successfully!")

In [ ]:
# Verify files on disk
print("Files in ../data/processed/:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    full_path = os.path.join(PROCESSED_DIR, f)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"  {f:<40} {size_kb:>8.1f} KB")

---
## Section 10: Final Conclusions

### What Was Achieved in This Notebook

We transformed raw cleaned datasets into three model-ready feature tables:

| Output File | Rows | Purpose |
|---|---|---|
| `user_features.csv` | ~1.3M | Describes each customer — feeds the User Tower |
| `item_features.csv` | ~105K | Describes each article — feeds the Item Tower |
| `interaction_features.csv` | ~31M+ | Training signal — (user, item, label) pairs |

### How These Features Connect to the Two-Tower Model

In the next phase (`04_model_training.ipynb`), we will build a Two-Tower model using **TensorFlow Recommenders (TFRS)**:

```
                      TWO-TOWER ARCHITECTURE

  user_features.csv         item_features.csv
        │                         │
        ▼                         ▼
  ┌───────────┐             ┌───────────┐
  │ User Tower│             │ Item Tower│
  │  (Dense   │             │  (Dense   │
  │   Layers) │             │   Layers) │
  └─────┬─────┘             └─────┬─────┘
        │                         │
        ▼                         ▼
   User Embedding            Item Embedding
   [128-dim vector]          [128-dim vector]
        │                         │
        └─────────┬───────────────┘
                  ▼
           Dot Product Score
           (high = recommend)

  Training signal: interaction_features.csv
  Loss: maximize score for label=1, minimize for label=0
```

### Why Notebooks vs. Production Scripts?

| | Notebooks (`notebooks/`) | Production Scripts (`src/`) |
|---|---|---|
| **Purpose** | Experimentation & exploration | Reliable, repeatable execution |
| **Audience** | You (the engineer, iterating) | CI/CD, teammates, schedules |
| **Format** | Interactive cells with visuals | Pure Python functions & modules |
| **Execution** | Manual (run cell by cell) | Automated (`python run_feature_pipeline.py`) |
| **Version control** | Harder (JSON diffs) | Clean Python diffs |

The notebook is where you **discover and validate ideas**. The production scripts are where those ideas **live permanently**.

---

### Next Steps

1. **Run the production pipeline** to generate features on the full dataset:
   ```bash
   python src/feature_engineering/run_feature_pipeline.py
   ```

2. **Proceed to model training:** Open `notebooks/04_model_training.ipynb`

3. **Potential improvements to explore:**
   - Add sequence features (last N items purchased, ordered by date)
   - Add channel/sales-channel features if available
   - Experiment with hard negative mining for better training signal
   - Add time-of-day and day-of-week purchase patterns